# 7단계: 연속 3시간(C) 최종 시험 + "도와주세요" 단어 확인(Whisper)

**학습은 하지 않고 평가만** 합니다. 한 번에 전부 실행해도 되고, 총 **20분~2시간**(GPU면 짧고 CPU면 깁니다)이 걸립니다. **GPU(T4) 런타임을 권장**합니다.

준비: 드라이브 `6_own\장시간 음성` 폴더에 3시간 녹음을 넣어 두세요. (`응급` 폴더의 "도와주세요" 녹음과 모델 파일 3개 `classifier_mlp_v2/v3_own/v4_own_all.keras`가 드라이브에 있어야 합니다.)

하는 일
1. C를 변환·임베딩하고 말소리 비율과 음량을 봅니다. (다른 기기로 녹음한 일상 소리)
2. 모델 3개(03b, 04, 05)의 **오경보 vs 위급 재현율 표** (재현율은 "도와주세요" 녹음, 오경보는 C)
3. Whisper로 말소리 구간의 **글자를 인식해서** "도와/살려/구해/119/신고" 단어가 나오는지 확인하고, 소리 모델과 합친 2단계 방식을 평가합니다.

**5번, 6번, 7번 항목의 출력**을 복사해서 알려 주세요. (결과 표는 드라이브에 CSV로도 저장됩니다)

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. YAMNet 불러오기

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
with tf.io.gfile.GFile(yamnet.class_map_path().numpy()) as f:
    names = [row["display_name"] for row in csv.DictReader(f)]
SPEECH = names.index("Speech")
SCREAM = [names.index(n) for n in ("Screaming", "Shout", "Yell")]
print(len(names), "클래스")

## 3. 새 녹음 변환·임베딩 (C가 있으면 몇 분)
이미 처리한 파일은 캐시를 씁니다. 출력에서 C의 길이, 말소리 비율, 음량을 확인하세요.

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))


PKL = f"{OUT_DIR}/own_embeddings_v1.pkl"
own = pickle.load(open(PKL, "rb")) if os.path.exists(PKL) else {}
nfc = lambda s: unicodedata.normalize("NFC", s)
for v in own.values():
    v["split"], v["name"] = nfc(v["split"]), nfc(v["name"])

for f in files:
    if f in own:
        continue
    x, how, corr = load_mono(f)
    emb, sp, sc, db, ts = embed_stream(x)
    own[f] = dict(split=nfc(os.path.basename(os.path.dirname(f))), name=nfc(os.path.basename(f)), emb=emb, speech=sp,
                  scream=sc, db=db, t=ts, dur=len(x) / SR, how=how, corr=corr)
    print(f"새로 처리: {own[f]['split']}/{own[f]['name'][:30]} {own[f]['dur'] / 60:.1f}분 | {how}", flush=True)
pickle.dump(own, open(PKL, "wb"))

E_files = [f for f, v in own.items() if "응급" in v["split"]]
C_files = [f for f, v in own.items() if ("장시간" in v["split"]) or ("연속" in v["name"]) or ("3시간" in v["name"])]
assert E_files, "응급 폴더의 녹음을 못 찾았어요."
for f in E_files:
    print(f"응급 소리: {own[f]['name']} {own[f]['dur']:.0f}초")
for f in C_files:
    v = own[f]; db = v["db"]
    print(f"C: {v['name']} {v['dur'] / 3600:.2f}시간 | {v['how']} | 말소리 {float((v['speech'] > 0.5).mean()):.0%} "
          f"(약 {float((v['speech'] > 0.5).mean()) * v['dur'] / 60:.0f}분) | 음량 중앙 {np.median(db):.0f} / 큰소리 {np.percentile(db, 95):.0f} dBFS "
          f"| 비명류(>0.3) {float((v['scream'] > 0.3).mean()):.1%}")
if not C_files:
    print("※ C(장시간 음성 폴더 또는 이름에 '연속'/'3시간')를 못 찾았어요. 오경보 부분은 건너뜁니다.")

## 4. 평가 도구
말소리가 잡힌 덩어리를 발화 하나로 보고(재현율), 알림은 "N프레임 연속 + 쉬는 시간" 규칙으로 셉니다.

In [ ]:
from tensorflow import keras
SPEECH_TH = 0.3

def alarm_idx(p, t, N, thr, cooldown):
    ok = (p >= thr).astype(int)
    if len(ok) < N:
        return []
    run = np.convolve(ok, np.ones(N, int), "valid") == N
    out, last = [], -1e9
    for i in np.flatnonzero(run) + (N - 1):
        if t[i] - last >= cooldown:
            out.append(int(i)); last = t[i]
    return out

def at(p, v, N, thr, cooldown):          # 알림 시각(연속 구간의 시작)
    return [v["t"][i - (N - 1)] for i in alarm_idx(p, v["t"], N, thr, cooldown)]

def speech_windows(v, gap):              # 말소리가 잡힌 덩어리 (시작, 끝)
    sp, t = v["speech"].astype(np.float32), v["t"]
    out = []
    for i in np.flatnonzero(sp > SPEECH_TH):
        if out and t[i] - t[out[-1][1]] <= gap:
            out[-1][1] = i
        else:
            out.append([i, i])
    return [(float(t[a]), float(t[b]) + 0.96) for a, b in out]

mmss = lambda s: f"{int(s) // 3600}:{int(s) % 3600 // 60:02d}:{int(s) % 60:02d}"
E_cl = {f: speech_windows(own[f], 1.5) for f in E_files}
n_E = sum(len(c) for c in E_cl.values())
hoursC = sum(own[f]["dur"] for f in C_files) / 3600
print(f"응급 소리 발화 덩어리 {n_E}개 (파일 {len(E_files)}개), C {hoursC:.2f}시간")

MODELS = [("03b", "classifier_mlp_v2.keras"), ("04", "classifier_mlp_v3_own.keras"), ("05", "classifier_mlp_v4_own_all.keras")]
P = {}
for label, fn in MODELS:
    try:
        m = keras.models.load_model(find(fn))
    except AssertionError as e:
        print("건너뜀:", e); continue
    P[label] = {f: m.predict(own[f]["emb"].astype(np.float32), batch_size=4096, verbose=0).ravel() for f in E_files + C_files}
print("모델:", list(P))

## 5. 오경보 vs 위급 재현율 (소리 모델만)
- **E재현율**: "도와주세요" 녹음의 발화 중 앞뒤 1초 안에 알림이 울린 비율 (쉬는 시간 1초)
- **C오경보 / C시간당**: 연속 3시간(다른 기기)에서 울린 알림 (쉬는 시간 5초). 위급 소리가 우연히 들어 있었다면 그만큼 오경보가 과대 계산됩니다.

In [ ]:
pd.set_option("display.width", 200)
rows = []
for label in P:
    for N in (1, 2):
        for thr in (0.5, 0.8, 0.9, 0.95, 0.98):
            hit = sum(any(c0 - 1.0 <= x <= c1 + 1.0 for x in at(P[label][f], own[f], N, thr, 1.0)) for f in E_files for c0, c1 in E_cl[f])
            r = {"모델": label, "N": N, "임계값": thr, "E감지": f"{hit}/{n_E}", "E재현율": hit / max(n_E, 1)}
            if C_files:
                n = sum(len(alarm_idx(P[label][f], own[f]["t"], N, thr, 5.0)) for f in C_files)
                r["C오경보(횟수)"], r["C시간당"] = n, n / hoursC
            rows.append(r)
tradeoff = pd.DataFrame(rows)
print(tradeoff.round(3).to_string(index=False))
tradeoff.to_csv(f"{OUT_DIR}/result_tradeoff.csv", index=False, encoding="utf-8-sig")

print("\n[파일별 E재현율] 모델 05·N=1·0.5 / 04·N=1·0.5 / 03b·N=1·0.5")
for f in E_files:
    cl = E_cl[f]
    out = []
    for label in ("05", "04", "03b"):
        if label in P:
            ts = at(P[label][f], own[f], 1, 0.5, 1.0)
            out.append(f"{label} {sum(any(c0 - 1 <= x <= c1 + 1 for x in ts) for c0, c1 in cl)}/{len(cl)}")
    print(f"  {own[f]['name'][:34]}: " + " | ".join(out))

## 6. Whisper로 "도와주세요" 단어 확인 (2단계 방식 시험)
말소리가 잡힌 구간을 글자로 바꾸고 **도와/살려/구해/119/신고** 같은 단어가 나오는지 봅니다.
- **Whisper 단독**: 단어만으로 판단했을 때의 재현율과 C 오경보
- **소리 모델 + 단어**: 소리 모델이 울리고 **그 구간에 단어도 있을 때만** 위급으로 본 경우
- 구간 수와 처리 시간을 출력하니, 파이에서 가능한지 판단하는 자료가 됩니다. (런타임이 GPU면 빠릅니다)

In [ ]:
import re, time, torch
from transformers import pipeline

WHISPER = "openai/whisper-small"
KEY = re.compile(r"도와\s*(주|줘|달|요|세)|살려|구해|119|일일구|신고|사람\s*살")
dev = 0 if torch.cuda.is_available() else -1
print("런타임:", "GPU" if dev == 0 else "CPU (오래 걸릴 수 있어요)")
asr = pipeline("automatic-speech-recognition", model=WHISPER, device=dev,
               torch_dtype=torch.float16 if dev == 0 else torch.float32)
GEN = {"language": "ko", "task": "transcribe", "max_new_tokens": 48}

def make_windows(v, gap):
    out = []
    for k, (c0, c1) in enumerate(speech_windows(v, gap)):
        a, w1 = max(0.0, c0 - 0.5), min(v["dur"], c1 + 0.5)
        while True:
            b = min(a + 10.0, w1)
            out.append((k, a, b))
            if b >= w1 - 1e-6:
                break
            a += 9.0
    return out

def transcribe(x, wins, tag):
    texts, t0 = [], time.time()
    for s in range(0, len(wins), 128):
        batch = [{"raw": x[int(a * SR):int(b * SR)].astype(np.float32) / 32768, "sampling_rate": SR} for _, a, b in wins[s:s + 128]]
        texts += [r["text"] for r in asr(batch, batch_size=16, generate_kwargs=GEN)]
        print(f"  {tag}: {min(s + 128, len(wins))}/{len(wins)} ({time.time() - t0:.0f}초)", flush=True)
    return texts, time.time() - t0

res = {}      # res[f] = (windows, texts)
tot_sec = tot_n = 0.0
for f in E_files + C_files:
    v = own[f]
    x, _, _ = load_mono(f)
    wins = make_windows(v, 1.5 if f in E_files else 1.0)
    print(f"{v['name'][:30]}: 말소리 구간 {len(wins)}개 (합계 {sum(b - a for _, a, b in wins) / 60:.1f}분)", flush=True)
    texts, sec = transcribe(x, wins, v["name"][:12])
    res[f] = (wins, texts); tot_sec += sec; tot_n += len(wins)
    del x
print(f"\nWhisper 처리: 구간 {int(tot_n)}개, 평균 {tot_sec / max(tot_n, 1):.2f}초/구간 ({'GPU' if dev == 0 else 'CPU'})")

def sound_ok(label, N, thr, f, a, b):
    return any(a - 1.0 <= x <= b + 1.0 for x in at(P[label][f], own[f], N, thr, 1.0))

def e_recall(cond):
    hit = 0
    for f in E_files:
        wins, texts = res[f]
        for k in range(len(E_cl[f])):
            hit += any(cond(f, a, b, tx) for (kk, a, b), tx in zip(wins, texts) if kk == k)
    return hit

def c_count(cond):
    n = 0
    for f in C_files:
        wins, texts = res[f]
        n += sum(bool(cond(f, a, b, tx)) for (_, a, b), tx in zip(wins, texts))
    return n

combos = [("Whisper 단독", lambda f, a, b, tx: bool(KEY.search(tx)))]
for label in P:
    for N, thr in ((1, 0.5), (1, 0.8), (2, 0.5)):
        combos.append((f"{label}·N={N}·{thr} + 단어",
                       (lambda lb, N_, th: lambda f, a, b, tx: bool(KEY.search(tx)) and sound_ok(lb, N_, th, f, a, b))(label, N, thr)))
out = []
for name, cond in combos:
    r = {"방식": name, "E재현율": f"{e_recall(cond)}/{n_E}"}
    if C_files:
        n = c_count(cond); r["C오경보(횟수)"], r["C시간당"] = n, round(n / hoursC, 2)
    out.append(r)
whisper_tbl = pd.DataFrame(out)
print(whisper_tbl.to_string(index=False))
whisper_tbl.to_csv(f"{OUT_DIR}/result_whisper.csv", index=False, encoding="utf-8-sig")

## 7. 직접 들어 볼 목록
- **응급 소리**: 발화마다 Whisper가 알아들은 글자와 단어 인식 여부, 소리 모델(05·04·03b, N=1·0.5)의 감지 O/X
- **C에서 단어가 인식된 구간**: 위급 단어가 나온 시각과 글자 (정상 대화에서 나온 것이면 2단계의 오경보 원인입니다)

In [ ]:
print("[응급 소리] 번호 시각 | 단어 | 05 04 03b | 인식한 글자")
k = 0
for f in E_files:
    wins, texts = res[f]
    for j in range(len(E_cl[f])):
        k += 1
        ws = [(a, b, tx) for (kk, a, b), tx in zip(wins, texts) if kk == j]
        a0 = ws[0][0]; txt = " ".join(tx.strip() for _, _, tx in ws)
        kw = "O" if any(KEY.search(tx) for _, _, tx in ws) else "X"
        fl = " ".join(("O" if any(sound_ok(lb, 1, 0.5, f, a, b) for a, b, _ in ws) else "X") if lb in P else "-" for lb in ("05", "04", "03b"))
        print(f"{k:2d}. {mmss(a0)[2:]} | {kw} | {fl} | {txt[:36]}")

for f in C_files:
    wins, texts = res[f]
    hits = [(a, b, tx) for (_, a, b), tx in zip(wins, texts) if KEY.search(tx)]
    print(f"\n[C: 단어 인식 {len(hits)}회 / {own[f]['dur'] / 3600:.2f}시간] 시각 | 소리모델 05 04 03b (N=1·0.5) | 글자")
    for a, b, tx in hits[:60]:
        fl = " ".join(("O" if sound_ok(lb, 1, 0.5, f, a, b) else "X") if lb in P else "-" for lb in ("05", "04", "03b"))
        print(f"  {mmss(a)} | {fl} | {tx.strip()[:40]}")
    if len(hits) > 60:
        print(f"  ... 이하 {len(hits) - 60}회 생략")